In [ ]:
from feast import FeatureStore
fs_oidc = FeatureStore(fs_yaml_file='/opt/app-root/src/feast-config/driver_ranking_oidc')

In [ ]:
auth_config = fs_oidc.config.auth_config
auth_type = getattr(auth_config, 'auth_type', None) or getattr(auth_config, 'type', None)
assert auth_type == 'oidc', f"Expected auth type 'oidc', got: '{auth_type}'"
print(f"✅ Auth type verified from FeatureStore config: {auth_type}")

In [ ]:
# Verify project
project_name = "driver_ranking_oidc"
project = fs_oidc.get_project(project_name)

assert project is not None, f"get_project('{project_name}') returned None"

if isinstance(project, dict):
    returned_name = project.get("spec", {}).get("name")
else:
    returned_name = getattr(project, "name", None)
    if not returned_name and hasattr(project, "spec") and hasattr(project.spec, "name"):
        returned_name = project.spec.name

assert returned_name, f"Returned project does not contain a valid name: {project}"
assert returned_name == project_name, (
    f"Expected project '{project_name}', but got '{returned_name}'"
)

print(f"get_project('{project_name}') validation passed!")

In [ ]:
# Validate get_entity
entity = fs_oidc.get_entity("driver")
assert entity is not None, "Entity 'driver' not found in registry"
assert entity.name == "driver", f"Entity name mismatch: {entity.name}"

join_keys = getattr(entity, "join_keys", None) or [getattr(entity, "join_key", "")]
assert "driver_id" in join_keys, f"Expected join_key 'driver_id', got: {join_keys}"
print(f"✅ get_entity('driver'): name={entity.name}, join_keys={join_keys}")

In [ ]:
# driver_hourly_stats_fresh is the registered view (PushSource-backed)
fv = fs_oidc.get_feature_view("driver_hourly_stats_fresh")
assert fv is not None, "FeatureView 'driver_hourly_stats_fresh' not found in registry"
assert fv.name == "driver_hourly_stats_fresh", f"FeatureView name mismatch: {fv.name}"
feature_names = [f.name for f in fv.features]
for expected in ["conv_rate", "acc_rate", "avg_daily_trips",
                 "driver_metadata", "driver_config", "driver_profile"]:
    assert expected in feature_names, f"'{expected}' not in features: {feature_names}"
print(f"✅ get_feature_view('driver_hourly_stats_fresh'): {len(fv.features)} features — {feature_names}")

In [ ]:
# Validate get_data_source
ds = fs_oidc.get_data_source("driver_hourly_stats_source")
assert ds is not None, "DataSource 'driver_hourly_stats_source' not found in registry"
assert ds.name == "driver_hourly_stats_source", f"DataSource name mismatch: {ds.name}"
print(f"✅ get_data_source('driver_hourly_stats_source'): {ds.name}")

In [ ]:
# Validate get_feature_service
for fs_name in ["driver_activity_v1", "driver_activity_v2", "driver_activity_v3"]:
    svc = fs_oidc.get_feature_service(fs_name)
    assert svc is not None, f"FeatureService '{fs_name}' not found in registry"
    assert svc.name == fs_name, f"FeatureService name mismatch: {svc.name}"
    print(f"✅ get_feature_service('{fs_name}'): {len(svc.feature_view_projections)} projections")

In [ ]:
# Validate get_on_demand_feature_view
for odfv_name in ["transformed_conv_rate", "transformed_conv_rate_fresh"]:
    odfv = fs_oidc.get_on_demand_feature_view(odfv_name)
    assert odfv is not None, f"OnDemandFeatureView '{odfv_name}' not found"
    if isinstance(odfv, dict):
        odfv_returned_name = odfv.get("spec", {}).get("name")
    else:
        odfv_returned_name = getattr(odfv, "name", None)
    assert odfv_returned_name == odfv_name, (
        f"OnDemandFeatureView name mismatch: {odfv_returned_name}"
    )
    odfv_features = [f.name for f in getattr(odfv, "features", [])]
    for expected in ["conv_rate_plus_val1", "conv_rate_plus_val2"]:
        assert expected in odfv_features, (
            f"'{expected}' not in {odfv_name} features: {odfv_features}"
        )
    print(f"✅ get_on_demand_feature_view('{odfv_name}'): features={odfv_features}")

In [ ]:
# Validate all list_* methods with OIDC auth.
non_empty_required = {
    "list_projects",
    "list_entities",
    "list_feature_views",
    "list_all_feature_views",
    "list_batch_feature_views",
    "list_on_demand_feature_views",
    "list_feature_services",
    "list_data_sources",
}

feast_list_functions = [
    "list_projects",
    "list_entities",
    "list_feature_views",
    "list_all_feature_views",
    "list_batch_feature_views",
    "list_on_demand_feature_views",
    "list_feature_services",
    "list_data_sources",
    "list_saved_datasets",
]

def validate_list_method(fs_obj, method_name, require_non_empty=False):
    assert hasattr(fs_obj, method_name), f"Method not found: {method_name}"
    result = getattr(fs_obj, method_name)()
    assert isinstance(result, list), (
        f"{method_name}() must return a list, got {type(result)}"
    )
    if require_non_empty:
        assert len(result) > 0, (
            f"{method_name}() returned an empty list — expected pre-applied data"
        )
    print(f"✅ {method_name}() returned {len(result)} items")

for m in feast_list_functions:
    validate_list_method(fs_oidc, m, require_non_empty=(m in non_empty_required))

print("\nAll list_* methods validated successfully with OIDC auth")

In [ ]:
import time
import pandas as pd

ts = pd.Timestamp.now(tz="UTC")
online_write_df = pd.DataFrame({
    "driver_id": [1001],
    "conv_rate": [0.75],
    "acc_rate": [0.91],
    "avg_daily_trips": [42],
    "driver_metadata": [{"region": "west"}],
    "driver_config": [{"level": "senior"}],
    "driver_profile": [{"name": "John", "age": "30"}],
    "event_timestamp": [ts],
    "created": [ts],
})

fs_oidc.write_to_online_store(
    feature_view_name="driver_hourly_stats_fresh",
    df=online_write_df,
    allow_registry_cache=False,
)
print("✅ write_to_online_store('driver_hourly_stats_fresh') succeeded")

response = None
last_error = None
for attempt in range(1, 4):
    try:
        response = fs_oidc.get_online_features(
            features=[
                "driver_hourly_stats_fresh:conv_rate",
                "driver_hourly_stats_fresh:acc_rate",
                "driver_hourly_stats_fresh:avg_daily_trips",
            ],
            entity_rows=[{"driver_id": 1001}],
        ).to_dict()
        break
    except Exception as exc:
        last_error = exc
        if attempt < 3:
            print(f"⚠️ Online read attempt {attempt}/3: {exc}")
            time.sleep(5)
        else:
            raise

assert response is not None, f"Online feature read failed: {last_error}"
assert "conv_rate" in response, f"'conv_rate' missing: {list(response.keys())}"
assert "acc_rate" in response, f"'acc_rate' missing: {list(response.keys())}"
assert response["avg_daily_trips"] == [42], (
    f"avg_daily_trips mismatch: {response['avg_daily_trips']}"
)
print(f"✅ get_online_features(): conv_rate={response['conv_rate']}, "
      f"acc_rate={response['acc_rate']}, avg_daily_trips={response['avg_daily_trips']}")

In [ ]:
# Validate list_permissions returns the OIDC admin group permission
permissions = fs_oidc.list_permissions()
assert isinstance(permissions, list), (
    f"list_permissions() must return a list, got {type(permissions)}"
)
assert len(permissions) > 0, "list_permissions() returned empty — expected admin_group_permission"

permission_names = [p.name for p in permissions]
print(f"Permissions found: {permission_names}")

assert "admin_group_permission" in permission_names, (
    f"Expected 'admin_group_permission' in permissions, but got: {permission_names}"
)
print("✅ admin_group_permission exists in Feast OIDC permissions")

In [ ]:
# Validate get_historical_features with OIDC admin auth.
from datetime import datetime, timezone
import pandas as pd

entity_df = pd.DataFrame({
    "driver_id": [1001, 1002, 1003],
    "event_timestamp": [datetime.now(tz=timezone.utc)] * 3,
})

job = fs_oidc.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats_fresh:conv_rate",
        "driver_hourly_stats_fresh:acc_rate",
        "driver_hourly_stats_fresh:avg_daily_trips",
    ],
)
training_df = job.to_df()

assert training_df is not None, "get_historical_features() returned None"
for col in ["conv_rate", "acc_rate", "avg_daily_trips"]:
    assert col in training_df.columns, (
        f"'{col}' missing from historical features: {list(training_df.columns)}"
    )
print(f"✅ get_historical_features(): {len(training_df)} rows")
print(f"   Columns: {list(training_df.columns)}")

In [ ]:
# Validate materialize_incremental with OIDC admin auth.

from datetime import datetime, timezone

end_date = datetime.now(tz=timezone.utc)

fs_oidc.materialize_incremental(
    end_date=end_date,
    feature_views=["driver_hourly_stats_fresh"],
)
print(f"✅ materialize_incremental(end_date={end_date.isoformat()}) succeeded with OIDC admin auth")

In [ ]:
from feast.errors import (
    FeatureServiceNotFoundException,
    FeatureViewNotFoundException,
    EntityNotFoundException,
)

# delete_feature_service
for fs_name in ["driver_activity_v1", "driver_activity_v2", "driver_activity_v3"]:
    fs_oidc.delete_feature_service(fs_name)
    remaining = [s.name for s in fs_oidc.list_feature_services()]
    assert fs_name not in remaining, (
        f"FeatureService '{fs_name}' still present after deletion"
    )
    try:
        fs_oidc.get_feature_service(fs_name)
        raise AssertionError(f"Expected get_feature_service('{fs_name}') to raise after deletion")
    except FeatureServiceNotFoundException:
        print(f"✅ delete_feature_service('{fs_name}'): confirmed FeatureServiceNotFoundException")

# delete_feature_view
for fv_name in ["driver_hourly_stats_fresh"]:
    fs_oidc.delete_feature_view(fv_name)
    remaining = [fv.name for fv in fs_oidc.list_feature_views()]
    assert fv_name not in remaining, (
        f"FeatureView '{fv_name}' still present after deletion"
    )
    try:
        fs_oidc.get_feature_view(fv_name)
        raise AssertionError(f"Expected get_feature_view('{fv_name}') to raise after deletion")
    except FeatureViewNotFoundException:
        print(f"✅ delete_feature_view('{fv_name}'): confirmed FeatureViewNotFoundException")

In [ ]:
# Validate teardown with OIDC admin auth.

fs_oidc.teardown()
print("✅ teardown() succeeded with OIDC admin auth")

In [ ]:
# close() releases gRPC channels and registry connections.
fs_oidc.close()
print("✅ close() succeeded")